<a href="https://colab.research.google.com/github/kaskare/colab_notebooks/blob/main/Copy_of_Mutagenesis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Install PyNeuraLogic from PyPI

In [ ]:
# ! pip install neuralogic

In [ ]:
# import logging
# import sys

# logging.basicConfig(format='%(asctime)s | %(levelname)s : %(message)s', level=logging.INFO, stream=sys.stdout)

### Faster Data Download using Rsync
Instead of looping through hundreds of files with individual `scp` calls, we can use `rsync` to sync the entire remote directory to Colab in one session. We'll use `sshpass` to handle the password non-interactively.

In [ ]:
# !apt-get install -y sshpass

# import os
# import subprocess

# SSH_USER_HOST = "kassyask@login3.rci.cvut.cz"
# SSH_PASSWORD = ""
# REMOTE_BASE = "/home/kassyask/neuro-vector-symbolic-architectures-raven/data/I-RAVEN-10000/"
# LOCAL_DIR = "/content/sample_data/raven_data/"

# os.makedirs(LOCAL_DIR, exist_ok=True)

# CONSTELLATIONS = [
#     "center_single", "distribute_four", "distribute_nine",
#     "in_center_single_out_center_single", "in_distribute_four_out_center_single",
#     "left_center_single_right_center_single", "up_center_single_down_center_single"
# ]

# MAX_TRAIN_INSTANCES = 200
# MAX_TEST_INSTANCES = 20

# def run_command(cmd_str, input_data=None):
#     process = subprocess.run(cmd_str, shell=True, capture_output=True, text=True, input=input_data)
#     return process.stdout.splitlines(), process.stderr.splitlines(), process.returncode

# print(f"Starting limited download to {LOCAL_DIR}")

# for constellation in CONSTELLATIONS:
#     remote_constellation_path = os.path.join(REMOTE_BASE, constellation)
#     local_constellation_path = os.path.join(LOCAL_DIR, constellation)
#     os.makedirs(local_constellation_path, exist_ok=True)

#     print(f"\nProcessing constellation: {constellation}")

#     # List remote files
#     list_cmd = f"sshpass -p '{SSH_PASSWORD}' ssh -o StrictHostKeyChecking=no {SSH_USER_HOST} 'ls {remote_constellation_path}'"
#     stdout, stderr, _ = run_command(list_cmd)

#     all_remote_files = [f.strip() for f in stdout if f.strip()]

#     # Organize by base name
#     train_files = [f for f in all_remote_files if "_train." in f]
#     test_files = [f for f in all_remote_files if "_test." in f]

#     def select_files(file_list, limit):
#         bases = sorted(list(set(f.replace(".npz", "").replace(".xml", "") for f in file_list)))
#         selected = []
#         for b in bases[:limit]:
#             if f"{b}.npz" in file_list: selected.append(f"{b}.npz")
#             if f"{b}.xml" in file_list: selected.append(f"{b}.xml")
#         return selected

#     to_download = select_files(train_files, MAX_TRAIN_INSTANCES) + select_files(test_files, MAX_TEST_INSTANCES)

#     if to_download:
#         print(f"  Downloading {len(to_download)} files...")
#         files_input = "\n".join(to_download) + "\n"

#         # Corrected rsync command structure
#         rsync_cmd = (
#             f"sshpass -p '{SSH_PASSWORD}' rsync -avz --files-from=- "
#             f"-e 'ssh -o StrictHostKeyChecking=no' "
#             f"{SSH_USER_HOST}:{remote_constellation_path}/ {local_constellation_path}/"
#         )

#         stdout_r, stderr_r, rc = run_command(rsync_cmd, input_data=files_input)
#         if rc == 0:
#             print("  Success.")
#         else:
#             print(f"  Error (RC {rc}):")
#             for line in stderr_r: print(f"    {line}")
#     else:
#         print("  No files found.")

# print("\nDownload process finished.")

Vizualize the example

In [ ]:
# import numpy as np
# from PIL import Image, ImageDraw, ImageFont
# import xml.etree.ElementTree as ET

# # Define necessary variables for this cell
# save_path = "/content/I-RAVEN_samples/png.png"
# sample_data_path = "/content/sample_data/" # Ensure this is defined
# sample_data_prefix = f"{sample_data_path}/RAVEN_0_train"
# tree = ET.parse(f"{sample_data_prefix}.xml")
# root = tree.getroot()
# # Assuming a default config for visualization if not explicitly passed from elsewhere
# config = "in_center_single_out_center_single"
# pos_map = {
#     "center_single": {0: 0},
#     "distribute_four": {0: 1, 1: 2, 2: 3, 3: 4},
#     "distribute_nine": {0: 5, 1: 6, 2: 7, 3: 8, 4: 9, 5: 10, 6: 11, 7: 12, 8: 13},
#     "left_center_single_right_center_single": {0: 14, 1: 15},
#     "up_center_single_down_center_single": {0: 16, 1: 17},
#     "in_center_single_out_center_single": {0: 0, 1: 9},
#     "in_distribute_four_out_center_single": {0: 0, 1: 18, 2: 19, 3: 20, 4: 21},
# }

# def _get_font(size):
#     try:
#         return ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf", size)
#     except:
#         return ImageFont.load_default()

# # Re-adjusting the shape mapping based on user visual feedback
# # Current evidence:
# # - Objects drawn as pentagons are labeled as circles
# # - Objects drawn as hexagons are labeled as pentagons
# # - Objects drawn as circles are labeled as hexagons
# TYPE_NAMES = {0: "tri", 1: "sq", 2: "pent", 3: "hex", 4: "circ"}

# def panel_matches(pred, gt):
#     if len(pred) != len(gt): return False
#     p_sorted = sorted(pred, key=lambda x: x['position'])
#     g_sorted = sorted(gt, key=lambda x: x['position'])
#     for p, g in zip(p_sorted, g_sorted):
#         if p != g: return False
#     return True

# def parse_visual_data(root, config):
#     images_data = np.load(f"{sample_data_path}/RAVEN_0_train.npz")
#     images = images_data['image']
#     target = int(root.attrib.get('target', images_data.get('target', 0)))

#     all_panels_attributes = []
#     panels = root[0]

#     for panel in panels:
#         objs = []
#         global_offset = 0
#         for struct in panel:
#             for comp in struct:
#                 for layout in comp:
#                     pos_list = eval(layout.attrib["Position"])
#                     for entity in layout:
#                         bbox = eval(entity.attrib["bbox"])
#                         local_pos = pos_list.index(bbox) + global_offset
#                         objs.append({
#                             "type": int(entity.attrib["Type"]) - 1,
#                             "size": int(entity.attrib["Size"]),
#                             "color": int(entity.attrib["Color"]),
#                             "position": pos_map[config][local_pos]
#                         })
#                     global_offset += len(pos_list)
#         all_panels_attributes.append(objs)

#     return images, target, all_panels_attributes

# def visualize_sample(images, target, pred_panels, gt_panels, save_path, config=None):
#     H, W = images.shape[1], images.shape[2]
#     pad, ann_h, border = 6, 60, 3
#     font, font_sm = _get_font(11), _get_font(10)
#     cell_w, cell_h = W + pad, H + ann_h + pad

#     ctx_cols, ctx_rows = 3, 3
#     ctx_w, ctx_h = ctx_cols * cell_w - pad, ctx_rows * cell_h - pad
#     ans_cols, ans_w, ans_h = 8, 8 * cell_w - pad, cell_h

#     total_w = max(ctx_w, ans_w) + 2 * pad
#     gap_between, title_h = 16, 22
#     total_h = title_h + ctx_h + gap_between + ans_h + pad

#     canvas = Image.new("RGB", (total_w, total_h), (255, 255, 255))
#     draw = ImageDraw.Draw(canvas)
#     draw.text((pad, 2), f"Config: {config or 'unknown'}   |   Correct answer: candidate {target}", fill=(60, 60, 60), font=font)

#     def _paste_panel(panel_idx, x0, y0, highlight=False):
#         panel_img = Image.fromarray(images[panel_idx]).convert("RGB")
#         if highlight:
#             bordered = Image.new("RGB", (W + 2*border, H + 2*border), (0, 180, 0))
#             bordered.paste(panel_img, (border, border))
#             canvas.paste(bordered, (x0 - border, y0 - border))
#         else:
#             canvas.paste(panel_img, (x0, y0))

#         pred = pred_panels[panel_idx]
#         gt = gt_panels[panel_idx] if gt_panels else None
#         match = panel_matches(pred, gt) if gt else None

#         lines = [f"p{obj['position']} {TYPE_NAMES.get(obj['type'], 't'+str(obj['type']))} s{obj['size']} c{obj['color']}" for obj in pred] or ["[empty]"]
#         color = (0, 140, 0) if match else (200, 0, 0) if match is not None else (60, 60, 60)

#         ty = y0 + H + 2
#         for line in lines[:4]:
#             draw.text((x0, ty), line, fill=color, font=font_sm)
#             ty += 14

#     ctx_x0, ctx_y0 = (total_w - ctx_w) // 2, title_h
#     for row in range(ctx_rows):
#         for col in range(ctx_cols):
#             idx = row * 3 + col
#             x, y = ctx_x0 + col * cell_w, ctx_y0 + row * cell_h
#             if idx < 8: _paste_panel(idx, x, y)
#             else:
#                 qimg = Image.new("RGB", (W, H), (180, 180, 180))
#                 ImageDraw.Draw(qimg).text((W//2-6, H//2-8), "?", fill=(80, 80, 80), font=_get_font(24))
#                 canvas.paste(qimg, (x, y))

#     ans_x0, ans_y0 = (total_w - ans_w) // 2, ctx_y0 + ctx_h + gap_between
#     for i in range(8): _paste_panel(8 + i, ans_x0 + i * cell_w, ans_y0, highlight=(i == target))

#     canvas.save(save_path)
#     display(canvas)

# imgs, tgt, attrs = parse_visual_data(root, config)
# visualize_sample(imgs, tgt, attrs, attrs, "raven_sample.png", config=config)

### 1. Parsing the Example and Building the Dataset

Imports & Constants


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 0.  IMPORTS & CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────
import xml.etree.ElementTree as ET
import numpy as np
import os
from neuralogic.core import R, V, Template, Settings
from neuralogic.dataset import Dataset
from neuralogic.nn import get_evaluator
from neuralogic.optim import Adam
from neuralogic.nn.loss import MSE

# ── Dataset constants ─────────────────────────────────────────────────────────
CONSTELLATIONS = [
    "center_single",
    "distribute_four",
    "distribute_nine",
    "in_center_single_out_center_single",
    "in_distribute_four_out_center_single",
    "left_center_single_right_center_single",
    "up_center_single_down_center_single",
]

LOCAL_DATA_DIR = "/home/kassyask/neuro-vector-symbolic-architectures-raven/data/I-RAVEN-10000"
NUM_CONTEXT    = 8        # 8 context panels arranged in a 3x3 grid (bottom-right missing)
NUM_CANDIDATES = 8        # 8 answer candidates
NUM_POSITIONS  = 22       # max distinct positions across all constellations
TYPE_RANGE     = range(1, 6)   # 5 shape types
SIZE_RANGE     = range(0, 7)   # 7 size levels
COLOR_RANGE    = range(0, 10)  # 10 colour levels
EMBED_DIM      = 64       # increased from 8 for richer representations

MAX_TRAIN = 400
MAX_TEST  = 40
EPOCHS    = 400
LOG_EVERY = 10

# ── Panel grid layout ──────────────────────────────────────────────────────────
# The 3x3 RPM matrix. Panel index → (row, col)
# Panels 0-7 are context; the 9th slot is the missing panel.
PANEL_TO_RC = {
    0: (0, 0), 1: (0, 1), 2: (0, 2),
    3: (1, 0), 4: (1, 1), 5: (1, 2),
    6: (2, 0), 7: (2, 1),
    # candidate occupies (2,2); stored as panel index 8
    8: (2, 2),
}

# Intra-panel position centers (used for spatial edge assignment)
POS_CENTERS = {i: (float(i % 6), float(i // 6)) for i in range(NUM_POSITIONS)}

# Map from constellation name → local-position-index → global-position-id
POS_MAP = {
    "center_single":                          {0: 0},
    "distribute_four":                        {0: 1, 1: 2, 2: 3, 3: 4},
    "distribute_nine":                        {i: i + 5 for i in range(9)},
    "left_center_single_right_center_single": {0: 14, 1: 15},
    "up_center_single_down_center_single":    {0: 16, 1: 17},
    "in_center_single_out_center_single":     {0: 0, 1: 9},
    "in_distribute_four_out_center_single":   {0: 0, 1: 18, 2: 19, 3: 20, 4: 21},
}

# Spatial relation labels
INTRA_SPATIAL = ["left_of_intra", "right_of_intra", "above_intra", "below_intra"]
INTER_ROW     = ["left_of_row", "right_of_row"]     # same-row inter-panel edges
INTER_COL     = ["above_col", "below_col"]           # same-col inter-panel edges
CORRESP_RELS  = ["row_corr", "col_corr"]             # positional-correspondence edges
ALL_EDGE_RELS = INTRA_SPATIAL + INTER_ROW + INTER_COL + CORRESP_RELS

print("Constants ready ✓")



Constants ready ✓


/home/kassyask/.conda/envs/nvsa/lib/python3.7/site-packages/neuralogic/core/builder/builder.py:4: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


Parsing & Sub-graph Construction

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 1.  GRAPH CONSTRUCTION
# ─────────────────────────────────────────────────────────────────────────────
#
# WHY A HIERARCHICAL GRAPH?
# ─────────────────────────
# An RPM problem has a 3-level structure:
#
#   OBJECT LEVEL  – individual shapes with type/size/colour/position attributes
#   PANEL LEVEL   – one cell of the 3x3 matrix; a set of objects
#   ROW/COL LEVEL – three panels in a row or column share abstract rules
#
# Standard flat GNNs (as in the baseline notebook) only model object-level
# message passing.  Because RPM rules like "Progression" or "Arithmetic"
# operate *across* panels (e.g. size increases left→right in a row), a flat
# GNN must discover these patterns implicitly through multi-hop paths.
#
# The improved design adds:
#   • panel_node(P)   – one aggregator node per panel
#   • row_node(R)     – one node per row (0,1,2)
#   • col_node(C)     – one node per column (0,1,2)
#
# And two new edge families:
#   • row_corr / col_corr edges  – directly connect objects that share the
#                                  same within-panel position across panels
#                                  in the same row / column.
#   • inter-panel spatial edges  – left_of_row / right_of_row / above_col /
#                                  below_col between adjacent panels
#
# ─────────────────────────────────────────────────────────────────────────────

def parse_panel(panel_elem, panel_idx, config_name):
    objects, obj_count = [], 0
    for struct in panel_elem:
        for comp in struct:
            for layout in comp:
                pos_list = eval(layout.attrib["Position"])
                for entity in layout:
                    obj_id     = f"p{panel_idx}_o{obj_count}"
                    local_idx  = pos_list.index(eval(entity.attrib["bbox"]))
                    mapped_pos = POS_MAP.get(config_name, {}).get(local_idx, 0)
                    objects.append((
                        obj_id, mapped_pos,
                        int(entity.attrib["Type"]),
                        int(entity.attrib["Size"]),
                        int(entity.attrib["Color"]),
                    ))
                    obj_count += 1
    return objects


def _intra_spatial_rels(sx, sy, dx, dy, eps=1e-4):
    rels = []
    if   sx < dx - eps: rels.append("left_of_intra")
    elif sx > dx + eps: rels.append("right_of_intra")
    if   sy < dy - eps: rels.append("above_intra")
    elif sy > dy + eps: rels.append("below_intra")
    return rels


def build_subgraph(panel_objects_list, cand_idx):
    """
    Node types (all objects — no panel/row/col nodes):
      obj_id  e.g. "p0_o0"

    Edge types:
      intra-panel spatial  — left_of_intra / right_of_intra / above_intra / below_intra
                             between every pair of objects within the same panel
                             O(n²) per panel, n ≤ 9, always bounded

      same_panel           — undirected, every pair of objects in the same panel
                             used in the template to give each object a view of
                             its whole panel without a panel aggregator node

      row_anchor / col_anchor — connects the anchor object (lowest position index)
                             of adjacent panels in the same row / column
                             O(1) per panel pair — zero grounding blowup
                             carries cross-panel progression signal

    Unary markers:
      is_cand(X)           — marks objects that belong to the candidate panel
      is_anchor(X)         — marks the anchor object of each panel
    """
    facts = []
    eid   = [0]

    def add_edge(ni, nj, rel):
        e = f"e{eid[0]}"
        eid[0] += 1
        facts.append(R.edge(ni, nj, e))
        facts.append(R.get(rel)(e))

    cand_panel_orig = NUM_CONTEXT + cand_idx
    active_orig     = list(range(NUM_CONTEXT)) + [cand_panel_orig]

    # Internal index: 0..7 = context panels, 8 = candidate panel
    def internal(orig):
        return orig if orig < NUM_CONTEXT else 8

    # ── 1. Attribute facts + collect objects per panel ──────────────────────
    panel_objs = {}   # internal_panel_idx → [(obj_id, pos), ...]

    for orig in active_orig:
        ip   = internal(orig)
        objs = panel_objects_list[orig]
        panel_objs[ip] = []
        for obj_id, pos, typ, sz, col in objs:
            facts.append(R.get(f"type_{typ}")(obj_id))
            facts.append(R.get(f"size_{sz}")(obj_id))
            facts.append(R.get(f"color_{col}")(obj_id))
            facts.append(R.get(f"pos_{pos}")(obj_id))
            panel_objs[ip].append((obj_id, pos))

    # Mark candidate objects
    for obj_id, _ in panel_objs[8]:
        facts.append(R.is_cand(obj_id))

    # Mark anchor objects (lowest position index per panel)
    anchors = {}  # internal_panel_idx → obj_id
    for ip, objs in panel_objs.items():
        if objs:
            anchor_obj = min(objs, key=lambda x: x[1])[0]
            anchors[ip] = anchor_obj
            facts.append(R.is_anchor(anchor_obj))

    # ── 2. Intra-panel spatial edges ────────────────────────────────────────
    for ip, objs in panel_objs.items():
        for i, (ni, pi) in enumerate(objs):
            for j, (nj, pj) in enumerate(objs):
                if i == j:
                    continue
                sx, sy = POS_CENTERS[pi]
                dx, dy = POS_CENTERS[pj]
                for rel in _intra_spatial_rels(sx, sy, dx, dy):
                    add_edge(ni, nj, rel)

    # ── 3. Same-panel edges ─────────────────────────────────────────────────
    # Connects every pair of objects within the same panel.
    # Lets the template give each object a full view of its panel
    # without an explicit panel aggregator node.
    for ip, objs in panel_objs.items():
        for i, (ni, _) in enumerate(objs):
            for j, (nj, _) in enumerate(objs):
                if i != j:
                    add_edge(ni, nj, "same_panel")

    # ── 4. Anchor edges across panels ───────────────────────────────────────
    # One edge per adjacent panel pair — O(1), no blowup.
    # row_anchor: connects anchors of panels in the same row
    # col_anchor: connects anchors of panels in the same column
    for ip_i in range(9):
        for ip_j in range(9):
            if ip_i == ip_j:
                continue
            if ip_i not in anchors or ip_j not in anchors:
                continue
            ri, ci = PANEL_TO_RC[ip_i]
            rj, cj = PANEL_TO_RC[ip_j]
            if ri == rj and abs(ci - cj) == 1:
                add_edge(anchors[ip_i], anchors[ip_j], "row_anchor")
            elif ci == cj and abs(ri - rj) == 1:
                add_edge(anchors[ip_i], anchors[ip_j], "col_anchor")

    return facts

print("Graph builder ready ✓")

Graph builder ready ✓


Build Datasets

In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
# 2.  DATASET LOADING
# ─────────────────────────────────────────────────────────────────────────────

def load_rpms(split, max_problems):
    rpms = []
    for constellation in CONSTELLATIONS:
        folder = os.path.join(LOCAL_DATA_DIR, constellation)
        if not os.path.exists(folder):
            continue
        xml_files = sorted(
            f for f in os.listdir(folder) if f.endswith(f"_{split}.xml")
        )[:max_problems]
        for f_xml in xml_files:
            base     = f_xml.replace(".xml", "")
            xml_path = os.path.join(folder, f_xml)
            npz_path = os.path.join(folder, base + ".npz")
            if not os.path.isfile(npz_path):
                continue
            tree   = ET.parse(xml_path)
            target = int(np.load(npz_path)["target"])
            panel_objects_list = [
                parse_panel(p, i, constellation)
                for i, p in enumerate(tree.getroot()[0])
            ]
            rpms.append((
                [build_subgraph(panel_objects_list, cid)
                 for cid in range(NUM_CANDIDATES)],
                target,
            ))
    return rpms


def rpms_to_dataset(rpms, oversample_correct=1):
    ds = Dataset()
    for subgraph_facts_list, target in rpms:
        for cid in range(NUM_CANDIDATES):
            label   = 1.0 if cid == target else 0.0
            repeats = oversample_correct if cid == target else 1
            for _ in range(repeats):
                ds.add_example(subgraph_facts_list[cid])
                ds.add_query(R.predict[label])
    return ds


print("Dataset loaders ready ✓")

Dataset loaders ready ✓


Template

In [ ]:


# ─────────────────────────────────────────────────────────────────────────────
# 3.  IMPROVED TEMPLATE
# ─────────────────────────────────────────────────────────────────────────────
#
# Architecture overview
# ─────────────────────
#
#  Layer 0 – Attribute embeddings (separate learned lookup per attr value)
#    type_emb(X), size_emb(X), color_emb(X), pos_emb(X)  →  fused h0(X)
#    edge_emb(E) from edge relation label
#
#  Layer 1 – Intra-panel GNN (objects talk to objects inside their panel)
#    h1(X) ← h0(Y) where edge(X,Y,E) is an INTRA_SPATIAL edge
#    + skip connection from h0
#
#  Layer 2 – Positional-correspondence GNN (objects talk across panels)
#    h2(X) ← h1(Y) where edge(X,Y,E) is a row_corr / col_corr edge
#    + skip connection from h1
#
#  Layer 3 – Panel aggregation
#    panel_rep(P) ← h2(X) where obj_panel(X,P)
#
#  Layer 4 – Row/col reasoning
#    row_rep(R) ← panel_rep(P) where panel_in_row(P,R)
#                 + inter-panel edge features
#    col_rep(C) ← panel_rep(P) where panel_in_col(P,C)
#
#  Scoring
#    predict ← h2(X) where is_cand(X)          [object-level score]
#             + panel_rep("panel_8")             [panel-level score]
#             + row_rep("row_2") + col_rep("col_2")  [global context]
#
# WHY SEPARATE INTRA vs INTER LAYERS?
# ─────────────────────────────────────
# Intra-panel edges encode *spatial* layout (e.g. triangle is left of square
# within one panel). Inter-panel / correspondence edges encode *temporal*
# progression (e.g. the object at position 0 grows larger panel-by-panel).
# Running these in separate layers lets the model build rich per-panel
# representations before reasoning across panels.
#
# WHY ROW/COL ABSTRACT NODES?
# ────────────────────────────
# RAVEN rules are row-wise or column-wise. An explicit row_rep node aggregates
# all three panels in a row, making it trivial for the model to detect
# monotonic trends. Without this, the GNN would need O(distance) hops to
# compare the first and last panel in a row.
#
# ─────────────────────────────────────────────────────────────────────────────
INTRA_SPATIAL = ["left_of_intra", "right_of_intra", "above_intra", "below_intra"]
ALL_EDGE_RELS = INTRA_SPATIAL + ["same_panel", "row_anchor", "col_anchor"]

def build_template(D=EMBED_DIM):
    template = Template()

    # ── Level 0: Attribute lookup embeddings ─────────────────────────────────
    template.add_rules([R.type_emb(V.X)[D,]  <= R.get(f"type_{v}")(V.X)  for v in TYPE_RANGE])
    template.add_rules([R.size_emb(V.X)[D,]  <= R.get(f"size_{v}")(V.X)  for v in SIZE_RANGE])
    template.add_rules([R.color_emb(V.X)[D,] <= R.get(f"color_{v}")(V.X) for v in COLOR_RANGE])
    template.add_rules([R.pos_emb(V.X)[D,]   <= R.get(f"pos_{v}")(V.X)   for v in range(NUM_POSITIONS)])
    template.add_rules([R.edge_emb(V.E)[D,]  <= R.get(rel)(V.E)           for rel in ALL_EDGE_RELS])

    # Fuse attribute embeddings → h0
    template += R.h0(V.X) <= (
        R.type_emb(V.X)[D, D],
        R.size_emb(V.X)[D, D],
        R.color_emb(V.X)[D, D],
        R.pos_emb(V.X)[D, D],
    )

    # ── Level 1: Intra-panel spatial message passing ─────────────────────────
    # Objects learn about the spatial layout within their panel.
    for rel in INTRA_SPATIAL:
        template += R.h1(V.X) <= (
            R.h0(V.Y)[D, D],
            R.edge(V.X, V.Y, V.E),
            R.get(rel)(V.E),
        )
    template += R.h1(V.X) <= R.h0(V.X)[D, D]   # skip connection

    # ── Level 2: Implicit panel aggregation via same_panel edges ─────────────
    # Each object aggregates all its panel-mates → h_panel encodes
    # "what panel am I in?" without any explicit panel node.
    # Works regardless of how many objects each panel contains.
    template += R.h_panel(V.X) <= (
        R.h1(V.Y)[D, D],
        R.edge(V.X, V.Y, V.E),
        R.get("same_panel")(V.E),
    )
    template += R.h_panel(V.X) <= R.h1(V.X)[D, D]   # include self / skip

    # ── Scoring head ──────────────────────────────────────────────────────────
    # Signal 1: Relational row score
    #   Compare candidate anchor's panel embedding to its row-neighbour's.
    #   h_panel already encodes the full panel content, so this is effectively
    #   a panel-vs-panel comparison without a panel node.
    # Keep only the relational signals
    template += R.predict[1, D] <= (
        R.h_panel(V.X)[D, D],
        R.h_panel(V.Y)[D, D],
        R.edge(V.X, V.Y, V.E),
        R.get("row_anchor")(V.E),
        R.is_cand(V.X),
        R.is_anchor(V.X),
    )
    template += R.predict[1, D] <= (
        R.h_panel(V.X)[D, D],
        R.h_panel(V.Y)[D, D],
        R.edge(V.X, V.Y, V.E),
        R.get("col_anchor")(V.E),
        R.is_cand(V.X),
        R.is_anchor(V.X),
    )


    return template

template = build_template()
print("Template ready ✓")
print(template)

Template ready ✓
{64} type_emb(X) :- type_1(X).
{64} type_emb(X) :- type_2(X).
{64} type_emb(X) :- type_3(X).
{64} type_emb(X) :- type_4(X).
{64} type_emb(X) :- type_5(X).
{64} size_emb(X) :- size_0(X).
{64} size_emb(X) :- size_1(X).
{64} size_emb(X) :- size_2(X).
{64} size_emb(X) :- size_3(X).
{64} size_emb(X) :- size_4(X).
{64} size_emb(X) :- size_5(X).
{64} size_emb(X) :- size_6(X).
{64} color_emb(X) :- color_0(X).
{64} color_emb(X) :- color_1(X).
{64} color_emb(X) :- color_2(X).
{64} color_emb(X) :- color_3(X).
{64} color_emb(X) :- color_4(X).
{64} color_emb(X) :- color_5(X).
{64} color_emb(X) :- color_6(X).
{64} color_emb(X) :- color_7(X).
{64} color_emb(X) :- color_8(X).
{64} color_emb(X) :- color_9(X).
{64} pos_emb(X) :- pos_0(X).
{64} pos_emb(X) :- pos_1(X).
{64} pos_emb(X) :- pos_2(X).
{64} pos_emb(X) :- pos_3(X).
{64} pos_emb(X) :- pos_4(X).
{64} pos_emb(X) :- pos_5(X).
{64} pos_emb(X) :- pos_6(X).
{64} pos_emb(X) :- pos_7(X).
{64} pos_emb(X) :- pos_8(X).
{64} pos_emb(X) :- p

Train

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.  GROUNDING
# ─────────────────────────────────────────────────────────────────────────────

settings    = Settings(optimizer=Adam(lr=1e-3), epochs=EPOCHS, error_function=MSE())
evaluator   = get_evaluator(template, settings)

print("Loading training data …")
rpms_train  = load_rpms("train", MAX_TRAIN)
train_ds    = rpms_to_dataset(rpms_train, oversample_correct=3)
built_train = evaluator.build_dataset(train_ds)

Loading training data …


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.  TRAINING
# ─────────────────────────────────────────────────────────────────────────────

print(f"Training for {EPOCHS} epochs …")
for epoch, (loss, n) in enumerate(evaluator.train(built_train, epochs=EPOCHS), 1):
    if epoch % LOG_EVERY == 0 or epoch == EPOCHS:
        print(f"  Epoch {epoch:4d}/{EPOCHS}  loss = {loss/n:.4f}")


Training for 400 epochs …
  Epoch   10/400  loss = 0.2957
  Epoch   20/400  loss = 0.2262
  Epoch   30/400  loss = 0.2224
  Epoch   40/400  loss = 0.2196
  Epoch   50/400  loss = 0.2192
  Epoch   60/400  loss = 0.2191
  Epoch   70/400  loss = 0.2192
  Epoch   80/400  loss = 0.2188
  Epoch   90/400  loss = 0.2176
  Epoch  100/400  loss = 0.2186
  Epoch  110/400  loss = 0.2177
  Epoch  120/400  loss = 0.2186
  Epoch  130/400  loss = 0.2184
  Epoch  140/400  loss = 0.2182
  Epoch  150/400  loss = 0.2170
  Epoch  160/400  loss = 0.2173
  Epoch  170/400  loss = 0.2181
  Epoch  180/400  loss = 0.2170
  Epoch  190/400  loss = 0.2167
  Epoch  200/400  loss = 0.2174


Evaluate

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# 5.  EVALUATION
# ─────────────────────────────────────────────────────────────────────────────

print("\nLoading test data …")
total_correct, total_rpms = 0, 0

for constellation in CONSTELLATIONS:
    folder = os.path.join(LOCAL_DATA_DIR, constellation)
    if not os.path.exists(folder):
        continue
    xml_files = sorted(
        f for f in os.listdir(folder) if f.endswith("_test.xml")
    )[:MAX_TEST]
    targets, rpms = [], []
    for f_xml in xml_files:
        base     = f_xml.replace(".xml", "")
        npz_path = os.path.join(folder, base + ".npz")
        if not os.path.isfile(npz_path):
            continue
        tree   = ET.parse(os.path.join(folder, f_xml))
        target = int(np.load(npz_path)["target"])
        targets.append(target)
        panel_objects_list = [
            parse_panel(p, i, constellation)
            for i, p in enumerate(tree.getroot()[0])
        ]
        rpms.append((
            [build_subgraph_improved(panel_objects_list, cid)
             for cid in range(NUM_CANDIDATES)],
            target,
        ))

    test_ds    = rpms_to_dataset(rpms)
    built_test = evaluator.build_dataset(test_ds)
    scores     = list(evaluator.test(built_test))

    correct = sum(
        int(np.argmax(scores[i * NUM_CANDIDATES:(i + 1) * NUM_CANDIDATES])) == tgt
        for i, tgt in enumerate(targets)
    )
    acc = correct / len(targets) if targets else 0.0
    print(f"  {constellation:50s}: {correct}/{len(targets)} = {acc:.3f}")
    total_correct += correct
    total_rpms    += len(targets)

if total_rpms:
    print(f"\n  Overall: {total_correct}/{total_rpms} = {total_correct/total_rpms:.3f}")
